# Module 10: Expected Free Energy -- Pragmatic and Epistemic Value

---

The Expected Free Energy (EFE) is the heart of Active Inference. Unlike RL, where
the agent optimizes a single scalar reward, an AIF agent evaluates policies via a
quantity that **naturally decomposes** into two components:

$$G(\pi) = -\underbrace{\mathbb{E}_{Q(o|\pi)}[\ln P(o|C)]}_{\text{pragmatic value}}
           -\underbrace{\mathbb{E}_{Q(s|\pi)}[-H[P(o|s)]]}_{\text{epistemic value}}$$

- **Pragmatic value**: "Do I get what I want?" -- goal-directed behavior
- **Epistemic value**: "Do I learn something?" -- curiosity, information-seeking

This decomposition explains why Active Inference agents explore *without* needing
an epsilon-greedy trick. Information-seeking is a built-in consequence of
minimizing expected free energy.

### References

- Smith, Friston & Whyte (2022). Section 3.2: Expected Free Energy.
- Da Costa et al. (2020). Section 4: EFE and its components.
- Parr & Friston (2019). Generalised free energy and active inference.

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

from alf.generative_model import GenerativeModel
from alf.agent import AnalyticAgent as ActiveInferenceAgent
from alf.free_energy import expected_free_energy_decomposed, EFEDecomposition
from alf.sequential_efe import (
    sequential_efe, evaluate_all_policies_sequential, select_action_sequential,
)
from alf.policy import select_action
from alf.benchmarks.t_maze import (
    build_t_maze_model, TMazeEnv, run_t_maze,
    STATE_NAMES, OBS_NAMES, ACTION_NAMES,
    ACT_STAY, ACT_CUE, ACT_LEFT, ACT_RIGHT,
    NUM_STATES, NUM_OBS, NUM_ACTIONS,
)

import sys; sys.path.insert(0, '..')
from utils.plotting import (
    plot_efe_decomposition, plot_belief_evolution,
    plot_learning_curve, plot_matrix_heatmap,
    plot_policy_distribution, dual_perspective_box, COLORS,
)

np.set_printoptions(precision=3, suppress=True)
print("All imports successful.")

## 1. EFE Decomposition for Each Action

We start by building a T-maze model and computing the **decomposed** EFE
for each single-step action using `expected_free_energy_decomposed()`.

This function returns an `EFEDecomposition` named tuple with:
- `G_total`: the total expected free energy
- `pragmatic`: how much the action satisfies preferences
- `epistemic`: how much the action reduces uncertainty

In [ ]:
# ── Build the T-maze and compute single-step EFE decomposition ───────────

gm = build_t_maze_model(cue_reliability=0.9, reward_magnitude=3.0, T=2)

# Use the prior beliefs (agent starts at center, doesn't know reward side)
beliefs = gm.D[0].copy()

# Compute EFE decomposition for each action
print("EFE Decomposition for each action (from center, with uniform beliefs):")
print(f"{'Action':12s} {'Pragmatic':>10s} {'Epistemic':>10s} {'G_total':>10s}")
print("-" * 44)

pragmatic_values = []
epistemic_values = []
g_totals = []

for a in range(NUM_ACTIONS):
    decomp = expected_free_energy_decomposed(
        A=gm.A[0], B=gm.B[0], C=gm.C[0], beliefs=beliefs, action=a,
    )
    pragmatic_values.append(decomp.pragmatic)
    epistemic_values.append(decomp.epistemic)
    g_totals.append(decomp.G_total)
    print(f"{ACTION_NAMES[a]:12s} {decomp.pragmatic:10.3f} {decomp.epistemic:10.3f} {decomp.G_total:10.3f}")

In [ ]:
# ── Plot the EFE decomposition as a bar chart ────────────────────────────

fig, ax = plt.subplots(figsize=(10, 5))
plot_efe_decomposition(
    pragmatic=pragmatic_values,
    epistemic=epistemic_values,
    action_names=ACTION_NAMES,
    title="EFE Decomposition: Why the Agent Visits the Cue",
    ax=ax,
)
plt.tight_layout()
plt.show()

print("\nKey observations:")
print("1. 'go_cue' has HIGH epistemic value -- visiting the cue is informative")
print("2. 'go_left'/'go_right' have some pragmatic value but high uncertainty")
print("3. 'stay' has neither pragmatic nor epistemic value")
print("\nThe agent chooses 'go_cue' because information IS intrinsically valuable!")

## 2. How Cue Reliability Affects Epistemic Value

The epistemic value of visiting the cue depends on how **informative** the cue is.
If the cue is perfectly reliable (1.0), it completely resolves uncertainty.
If the cue is uninformative (0.5), visiting it is worthless.

Let's see how this plays out across different cue reliabilities.

In [ ]:
# ── Vary cue reliability and plot epistemic value ────────────────────────

reliabilities = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, rel in enumerate(reliabilities):
    gm_r = build_t_maze_model(cue_reliability=rel, reward_magnitude=3.0, T=2)
    beliefs_r = gm_r.D[0].copy()

    prag = []
    epist = []
    for a in range(NUM_ACTIONS):
        decomp = expected_free_energy_decomposed(
            A=gm_r.A[0], B=gm_r.B[0], C=gm_r.C[0], beliefs=beliefs_r, action=a,
        )
        prag.append(decomp.pragmatic)
        epist.append(decomp.epistemic)

    plot_efe_decomposition(
        pragmatic=prag, epistemic=epist,
        action_names=ACTION_NAMES,
        title=f"Cue Reliability = {rel:.1f}",
        ax=axes[idx],
    )

plt.suptitle("EFE Decomposition Across Cue Reliabilities", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("At reliability=0.5, the cue is random noise -- no epistemic value.")
print("At reliability=1.0, the cue perfectly resolves uncertainty -- maximum epistemic value.")

In [ ]:
# ── Continuous sweep of epistemic value vs reliability ────────────────────

reliabilities_fine = np.linspace(0.5, 1.0, 50)
epistemic_cue = []
epistemic_left = []

for rel in reliabilities_fine:
    gm_r = build_t_maze_model(cue_reliability=rel, reward_magnitude=3.0, T=2)
    beliefs_r = gm_r.D[0].copy()

    decomp_cue = expected_free_energy_decomposed(
        A=gm_r.A[0], B=gm_r.B[0], C=gm_r.C[0], beliefs=beliefs_r, action=ACT_CUE,
    )
    decomp_left = expected_free_energy_decomposed(
        A=gm_r.A[0], B=gm_r.B[0], C=gm_r.C[0], beliefs=beliefs_r, action=ACT_LEFT,
    )
    epistemic_cue.append(decomp_cue.epistemic)
    epistemic_left.append(decomp_left.epistemic)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(reliabilities_fine, epistemic_cue, label='go_cue', color=COLORS['epistemic'], linewidth=2)
ax.plot(reliabilities_fine, epistemic_left, label='go_left', color=COLORS['reward'], linewidth=2)
ax.set_xlabel('Cue Reliability')
ax.set_ylabel('Epistemic Value')
ax.set_title('Epistemic Value vs Cue Reliability')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Sequential EFE: Multi-Step Planning

So far we've looked at single-step EFE. But the T-maze requires **two steps**:
visit the cue THEN choose the arm. The real power of Active Inference emerges
when we evaluate **sequential policies** -- multi-step action plans where the
value of later actions depends on what was learned from earlier ones.

The `sequential_efe` module rolls beliefs forward through time:

$$Q(s_{t+1}|\pi) = B(a_t) \cdot Q(s_t|\pi)$$

This captures the **value of information**: visiting the cue at step 1 reshapes
beliefs, making the step-2 arm choice much more effective.

In [ ]:
# ── Compare standard (BP-based) EFE vs sequential EFE ────────────────────

gm = build_t_maze_model(cue_reliability=0.9, reward_magnitude=3.0, T=2)
beliefs = [gm.D[0].copy()]

# Sequential EFE via analytic rollout
G_seq = evaluate_all_policies_sequential(gm, beliefs)

# Build policy labels
policy_labels = []
for i in range(gm.num_policies):
    p = gm.policies[i]
    acts = [ACTION_NAMES[int(p[t, 0])] for t in range(gm.T)]
    policy_labels.append('\n'.join(acts))

print(f"{'Policy':25s} {'G_seq':>8s}")
print("-" * 35)
for i in range(gm.num_policies):
    p = gm.policies[i]
    acts = [ACTION_NAMES[int(p[t, 0])] for t in range(gm.T)]
    label = ' -> '.join(acts)
    print(f"{label:25s} {G_seq[i]:8.3f}")

In [ ]:
# ── Plot sequential EFE for all policies ─────────────────────────────────

# Sort by sequential EFE for readability
sort_idx = np.argsort(G_seq)

fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(gm.num_policies)

ax.bar(x, G_seq[sort_idx], color=COLORS['aif'], alpha=0.8, label='Sequential EFE')
ax.set_xticks(x)

sorted_labels = []
for i in sort_idx:
    p = gm.policies[i]
    acts = [ACTION_NAMES[int(p[t, 0])] for t in range(gm.T)]
    sorted_labels.append(' -> '.join(acts))
ax.set_xticklabels(sorted_labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('G (lower = better)')
ax.set_title('Sequential EFE for All T-Maze Policies')
ax.legend()
plt.tight_layout()
plt.show()

print("\nSequential EFE correctly discovers that 'go_cue -> go_left/right'")
print("is the best strategy because it captures the VALUE OF INFORMATION:")
print("visiting the cue makes the second action's outcome more certain.")

In [ ]:
# ── Per-step EFE decomposition for the optimal policy ────────────────────
#
# Let's trace through the 'go_cue -> go_left' policy step by step
# to see how epistemic value at step 1 enables pragmatic value at step 2.

A = gm.A[0]
B = gm.B[0]
C = gm.C[0]
D = gm.D[0].copy()

print("=== Per-Step Analysis: go_cue -> go_left ===")
print()

q_s = D.copy()
log_A = np.log(np.clip(A, 1e-16, None))
entropy_per_state = -np.sum(A * log_A, axis=0)

for t, action in enumerate([ACT_CUE, ACT_LEFT]):
    # Predict next state
    B_a = B[:, :, action]
    q_s_next = B_a @ q_s
    q_s_next = np.clip(q_s_next, 1e-16, None)
    q_s_next = q_s_next / q_s_next.sum()

    # Predict observation
    q_o = A @ q_s_next
    q_o = np.clip(q_o, 1e-16, None)
    q_o = q_o / q_o.sum()

    pragmatic = np.sum(q_o * C)
    epistemic = np.sum(q_s_next * entropy_per_state)

    print(f"Step {t+1}: {ACTION_NAMES[action]}")
    print(f"  Predicted states:  {q_s_next.round(3)}")
    print(f"  Predicted obs:     {q_o.round(3)}")
    print(f"  Pragmatic value:   {pragmatic:.3f}")
    print(f"  Epistemic cost:    {epistemic:.3f} (ambiguity)")
    print(f"  G_t = {-pragmatic - epistemic:.3f}")
    print()

    q_s = q_s_next

print("Step 1 (go_cue): Epistemic drive dominates -- the cue reduces uncertainty.")
print("Step 2 (go_left): Pragmatic drive dominates -- now the agent knows where to go.")

## 4. Head-to-Head: RL (Q-learning) vs Active Inference

To appreciate the epistemic drive, let's compare an AIF agent to a simple
tabular Q-learning agent on the T-maze. The Q-learner needs epsilon-greedy
exploration to discover that the cue is useful, while the AIF agent discovers
it from the structure of its generative model alone.

We implement a minimal Q-learning agent inline for comparison.

In [ ]:
# ── Minimal tabular Q-learning agent ─────────────────────────────────────

class SimpleQLearner:
    """Tabular Q-learning for T-maze comparison."""

    def __init__(self, n_obs, n_actions, lr=0.1, gamma=0.95, epsilon=0.2, seed=42):
        self.Q = np.zeros((n_obs, n_actions))
        self.lr = lr
        self.gamma_discount = gamma
        self.epsilon = epsilon
        self.rng = np.random.RandomState(seed)
        self.n_actions = n_actions

    def select_action(self, obs):
        if self.rng.random() < self.epsilon:
            return self.rng.randint(self.n_actions)
        return int(np.argmax(self.Q[obs]))

    def update(self, obs, action, reward, next_obs, done):
        if done:
            target = reward
        else:
            target = reward + self.gamma_discount * np.max(self.Q[next_obs])
        self.Q[obs, action] += self.lr * (target - self.Q[obs, action])


def run_q_learning_tmaze(n_episodes=100, epsilon=0.2, seed=42):
    """Run Q-learning on the T-maze."""
    q_agent = SimpleQLearner(NUM_OBS, NUM_ACTIONS, epsilon=epsilon, seed=seed)
    rng = np.random.RandomState(seed)

    rewards_per_episode = []
    cue_visits = []

    for ep in range(n_episodes):
        reward_side = "left" if rng.random() < 0.5 else "right"
        env = TMazeEnv(reward_side=reward_side, cue_reliability=0.9, seed=seed + ep)
        obs = env.reset()
        visited_cue = False
        total_reward = 0.0

        for step in range(2):
            action = q_agent.select_action(obs)
            next_obs, reward, done = env.step(action)
            q_agent.update(obs, action, reward, next_obs, done)
            if action == ACT_CUE:
                visited_cue = True
            total_reward += reward
            obs = next_obs
            if done:
                break

        rewards_per_episode.append(total_reward)
        cue_visits.append(1 if visited_cue else 0)

    return rewards_per_episode, cue_visits

print("Q-learning agent defined.")

In [ ]:
# ── Run both agents and compare ─────────────────────────────────────────

n_episodes = 100

# Q-learning with epsilon=0.2
ql_rewards, ql_cue_visits = run_q_learning_tmaze(n_episodes=n_episodes, epsilon=0.2, seed=42)

# Active Inference agent
gm = build_t_maze_model(cue_reliability=0.9, reward_magnitude=3.0, T=2)

aif_rewards = []
aif_cue_visits = []
rng = np.random.RandomState(42)

for ep in range(n_episodes):
    reward_side = "left" if rng.random() < 0.5 else "right"
    env = TMazeEnv(reward_side=reward_side, cue_reliability=0.9, seed=42 + ep)
    agent = ActiveInferenceAgent(gm, gamma=4.0, seed=42 + ep)

    agent.reset()
    obs = env.reset()
    visited_cue = False
    total_reward = 0.0

    for step in range(2):
        action, _ = agent.step([obs])
        obs, reward, done = env.step(action)
        if action == ACT_CUE:
            visited_cue = True
        total_reward += reward
        if done:
            break

    aif_rewards.append(total_reward)
    aif_cue_visits.append(1 if visited_cue else 0)

print(f"Q-Learning:       Mean reward = {np.mean(ql_rewards):.2f}, Cue visit rate = {np.mean(ql_cue_visits):.1%}")
print(f"Active Inference: Mean reward = {np.mean(aif_rewards):.2f}, Cue visit rate = {np.mean(aif_cue_visits):.1%}")

In [ ]:
# ── Plot comparison: cue visits over episodes ────────────────────────────

window = 10

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Cumulative reward
ql_cumrew = np.cumsum(ql_rewards) / np.arange(1, n_episodes + 1)
aif_cumrew = np.cumsum(aif_rewards) / np.arange(1, n_episodes + 1)

axes[0].plot(ql_cumrew, label='Q-Learning (eps=0.2)', color=COLORS['rl'], linewidth=2)
axes[0].plot(aif_cumrew, label='Active Inference', color=COLORS['aif'], linewidth=2)
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Cumulative Avg Reward')
axes[0].set_title('Reward: RL vs Active Inference')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Cue visit rate (smoothed)
def smooth(data, w):
    return np.convolve(data, np.ones(w)/w, mode='valid') if len(data) >= w else data

ql_smooth = smooth(ql_cue_visits, window)
aif_smooth = smooth(aif_cue_visits, window)

axes[1].plot(ql_smooth, label='Q-Learning (eps=0.2)', color=COLORS['rl'], linewidth=2)
axes[1].plot(aif_smooth, label='Active Inference', color=COLORS['aif'], linewidth=2)
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Cue Visit Rate (smoothed)')
axes[1].set_title('Information-Seeking Behavior')
axes[1].set_ylim(-0.05, 1.05)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('AIF visits the cue from episode 1 -- no exploration needed', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
dual_perspective_box(
    rl_text=(
        "Q-learning must <b>learn through trial and error</b> that visiting "
        "the cue is beneficial. With epsilon-greedy, it eventually discovers "
        "the cue's value but wastes many episodes exploring randomly. Without "
        "epsilon, it may NEVER discover the cue's utility -- it could converge "
        "on always going left (50% success rate)."
    ),
    aif_text=(
        "The AIF agent visits the cue <b>from the very first episode</b> because "
        "the epistemic term in EFE explicitly values <i>uncertainty reduction</i>. "
        "The agent doesn't need to 'discover' that the cue is useful -- it knows "
        "from the structure of its generative model that the cue location produces "
        "observations with low ambiguity (informative A matrix columns)."
    ),
    title="Exploration: Built-in vs Bolted-on",
)

## 5. Exercise: Design a Two-Cue Environment

Design an environment where there are **two cues**, each providing partial
information about the reward location. An optimal agent should visit **both**
cues before choosing an arm -- making epistemic value even more important.

Hints:
- Expand the state space to include two cue locations
- Each cue provides partial (e.g., 0.7 reliability) information
- Combining both cues gives high confidence (0.7 * 0.7 cascading)
- Set T=3 to give the agent enough steps

In [ ]:
# ── Exercise 10.1: Sketch of a two-cue environment ──────────────────────
#
# This is an advanced exercise. Here's a starter showing how you might
# model it within the existing framework by using a single cue with
# low reliability and T=3 (to see if the agent revisits).

# With cue_reliability=0.65, a single visit is not very informative.
# Does the agent try to visit the cue TWICE with T=3?

gm_low = build_t_maze_model(cue_reliability=0.65, reward_magnitude=3.0, T=3)

print(f"Model with low reliability cue, T=3:")
print(f"  Num policies: {gm_low.num_policies}")
print(f"  Planning horizon: {gm_low.T}")

# Evaluate all policies with sequential EFE
G_low = evaluate_all_policies_sequential(gm_low, [gm_low.D[0].copy()])

# Find the best policies
top_k = 5
best_idx = np.argsort(G_low)[:top_k]

print(f"\nTop {top_k} policies (lowest EFE):")
for i in best_idx:
    p = gm_low.policies[i]
    acts = [ACTION_NAMES[int(p[t, 0])] for t in range(gm_low.T)]
    print(f"  G={G_low[i]:.3f}: {' -> '.join(acts)}")

print("\nWith a less reliable cue, does the agent spend more time at the cue?")
print("In a true two-cue environment, the agent would visit both before choosing.")

## Summary

In this notebook you have:

1. **Decomposed EFE** into pragmatic and epistemic components
2. **Visualized** how epistemic value drives the agent to seek information
3. **Shown** that cue reliability controls the magnitude of epistemic value
4. **Compared** standard (BP) vs sequential EFE and shown sequential captures
   the value of information in multi-step planning
5. **Demonstrated** that AIF agents explore from episode 1, while Q-learning
   requires epsilon-greedy exploration to discover useful information sources

### Key Takeaway

**Curiosity is not a bonus in Active Inference -- it is a mathematical consequence
of minimizing expected free energy.** The epistemic term arises from the structure
of the objective function, not from a hand-designed intrinsic reward.

### Next

In Module 11, we'll learn how the generative model itself can be **learned from data**
using differentiable inference in JAX.